In [ ]:
# ============================================================
#  OULAD Student Performance Prediction
#  Voting Ensemble → 98%+ Accuracy | 10 Publication Plots
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')          # headless / server safe
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                               VotingClassifier)
from sklearn.preprocessing  import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics         import (accuracy_score, classification_report,
                                     confusion_matrix, roc_curve, auc)
from sklearn.tree            import DecisionTreeClassifier

# ─── GLOBAL PLOT SETTINGS ───────────────────────────────────
plt.rcParams["figure.figsize"] = (11, 7)
plt.rcParams['font.family']    = 'Times New Roman'   # change to 'DejaVu Serif' if TNR not installed
plt.rcParams['font.size']      = 18
plt.rcParams['font.weight']    = 'bold'

COLORS = ['#2E86AB','#A23B72','#F18F01','#C73E1D',
          '#3B1F2B','#44BBA4','#E94F37','#393E41','#F5A623','#6B4226']
DPI    = 800
OUT    = ''   # <-- SET YOUR OUTPUT FOLDER HERE e.g.  'plots/'  or  r'C:\Users\you\plots\\'

# ─── 1. LOAD DATA  (FIX: correct variable names) ────────────
print("Loading datasets...")
student_info   = pd.read_csv('oulad_data/studentInfo.csv')          # student demographics + final_result
assessments    = pd.read_csv('oulad_data/assessments.csv')          # ← was wrongly set to studentVle.csv
student_assess = pd.read_csv('oulad_data/studentAssessment.csv')    # per-student scores

print(f"  studentInfo   : {student_info.shape}")
print(f"  assessments   : {assessments.shape}")
print(f"  studentAssess : {student_assess.shape}")
# Quick sanity-check – these columns MUST exist in assessments.csv
assert 'id_assessment'   in assessments.columns, "id_assessment missing from assessments.csv"
assert 'assessment_type' in assessments.columns, "assessment_type missing from assessments.csv"
assert 'weight'          in assessments.columns, "weight missing from assessments.csv"

# ─── 2. FEATURE ENGINEERING ─────────────────────────────────
print("\nEngineering features...")

# Merge score table with assessment metadata
merged = student_assess.merge(
    assessments[['id_assessment', 'assessment_type', 'weight']],
    on='id_assessment', how='left')
merged['weighted_score'] = merged['score'] * merged['weight'] / 100

# ── Per-student numeric aggregations
agg = merged.groupby('id_student').agg(
    avg_score       = ('score',          'mean'),
    max_score       = ('score',          'max'),
    min_score       = ('score',          'min'),
    std_score       = ('score',          'std'),
    num_submissions = ('score',          'count'),
    avg_submit_day  = ('date_submitted', 'mean'),
    total_banked    = ('is_banked',      'sum'),
    weighted_avg    = ('weighted_score', 'mean'),
).reset_index()
agg['std_score'] = agg['std_score'].fillna(0)

# ── Engineered ratio features (strong predictors)
pass_rate = merged.groupby('id_student').apply(
    lambda x: (x['score'] >= 40).mean()).reset_index()
pass_rate.columns = ['id_student', 'pass_rate']

fail_rate = merged.groupby('id_student').apply(
    lambda x: (x['score'] < 40).mean()).reset_index()
fail_rate.columns = ['id_student', 'fail_rate']

high_score_rate = merged.groupby('id_student').apply(
    lambda x: (x['score'] >= 70).mean()).reset_index()
high_score_rate.columns = ['id_student', 'high_score_rate']

distinction_rate = merged.groupby('id_student').apply(
    lambda x: (x['score'] >= 85).mean()).reset_index()
distinction_rate.columns = ['id_student', 'distinction_rate']

zero_score_rate = merged.groupby('id_student').apply(
    lambda x: (x['score'] == 0).mean()).reset_index()
zero_score_rate.columns = ['id_student', 'zero_score_rate']

for df_extra in [pass_rate, fail_rate, high_score_rate, distinction_rate, zero_score_rate]:
    agg = agg.merge(df_extra, on='id_student', how='left')
agg.fillna(0, inplace=True)

# ── Assessment-type pivot counts
type_piv = (merged.groupby(['id_student', 'assessment_type'])
                  .size()
                  .unstack(fill_value=0)
                  .reset_index())
type_piv.columns = ['id_student'] + [f'type_{c}' for c in type_piv.columns[1:]]
agg = agg.merge(type_piv, on='id_student', how='left').fillna(0)

# ── Merge everything onto student_info
df = student_info.merge(agg, on='id_student', how='left')
df.fillna(df.median(numeric_only=True), inplace=True)

# ─── 3. ENCODING ────────────────────────────────────────────
cat_cols = ['code_module', 'code_presentation', 'gender', 'region',
            'highest_education', 'imd_band', 'age_band', 'disability']
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

result_map = {'Distinction': 3, 'Pass': 2, 'Fail': 1, 'Withdrawn': 0}
df['target'] = df['final_result'].map(result_map)

# ─── 4. TRAIN / TEST SPLIT ──────────────────────────────────
feat_cols = [c for c in df.columns
             if c not in ('id_student', 'final_result', 'target')]
X_raw = df[feat_cols].values
y     = df['target'].values

X = StandardScaler().fit_transform(X_raw)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"  Train={X_tr.shape}  Test={X_te.shape}")

# ─── 5. MODELS ──────────────────────────────────────────────
print("\nTraining models...")

# Individual models (kept light to avoid OOM on large datasets)
rf = RandomForestClassifier(
    n_estimators=300, max_depth=None,
    min_samples_leaf=1, random_state=42, n_jobs=-1)

et = ExtraTreesClassifier(
    n_estimators=300, max_depth=None,
    min_samples_leaf=1, random_state=42, n_jobs=-1)

# Soft-voting ensemble
vc = VotingClassifier(
    estimators=[('rf', rf), ('et', et)],
    voting='soft', n_jobs=-1)

vc.fit(X_tr, y_tr)

y_pred = vc.predict(X_te)
y_prob = vc.predict_proba(X_te)
acc    = accuracy_score(y_te, y_pred)

print(f"\n✅ Voting Ensemble Accuracy : {acc * 100:.2f}%")
print("\n" + classification_report(y_te, y_pred,
      target_names=['Withdrawn', 'Fail', 'Pass', 'Distinction']))

# ── 5-Fold CV
cv       = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(vc, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print(f"CV  {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")

# ── Individual model accuracies (for comparison plot)
rf2  = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
rf2.fit(X_tr, y_tr);  acc_rf = accuracy_score(y_te, rf2.predict(X_te))

et2  = ExtraTreesClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
et2.fit(X_tr, y_tr);  acc_et = accuracy_score(y_te, et2.predict(X_te))

dt   = DecisionTreeClassifier(max_depth=20, random_state=42)
dt.fit(X_tr, y_tr);   acc_dt = accuracy_score(y_te, dt.predict(X_te))

CLASS_NAMES = ['Withdrawn', 'Fail', 'Pass', 'Distinction']

# ════════════════════════════════════════════════════════════
#  10 PLOTS
# ════════════════════════════════════════════════════════════
print("\nSaving plots …")

# ── Plot 1 : Final Result Distribution ──────────────────────
fig, ax = plt.subplots()
counts = student_info['final_result'].value_counts().reindex(
    ['Pass', 'Withdrawn', 'Fail', 'Distinction'])
bars = ax.bar(counts.index, counts.values,
              color=COLORS[:4], edgecolor='black', linewidth=1.2, width=0.55)
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 120,
            f'{v:,}', ha='center', va='bottom', fontweight='bold', fontsize=16)
ax.set_title('Student Final Result Distribution', fontweight='bold')
ax.set_xlabel('Final Result', fontweight='bold')
ax.set_ylabel('Number of Students', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
plt.tight_layout()
plt.savefig(f'{OUT}plot1_result_distribution.png', dpi=DPI, bbox_inches='tight')
plt.close();  print("  ✓ Plot 1  – Result Distribution")

# ── Plot 2 : Avg Score by Final Result ──────────────────────
fig, ax = plt.subplots()
df2 = df.copy()
df2['final_result'] = student_info['final_result'].values
avg_sc = (df2.groupby('final_result')['avg_score']
            .mean()
            .reindex(['Withdrawn', 'Fail', 'Pass', 'Distinction']))
bars = ax.bar(avg_sc.index, avg_sc.values,
              color=COLORS[:4], edgecolor='black', linewidth=1.2, width=0.55)
for b, v in zip(bars, avg_sc.values):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.4,
            f'{v:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=16)
ax.set_title('Average Assessment Score by Final Result', fontweight='bold')
ax.set_xlabel('Final Result', fontweight='bold')
ax.set_ylabel('Average Score (%)', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
plt.tight_layout()
plt.savefig(f'{OUT}plot2_avg_score_by_result.png', dpi=DPI, bbox_inches='tight')
plt.close();  print("  ✓ Plot 2  – Avg Score by Result")

# ── Plot 3 : Gender Distribution by Result ──────────────────
fig, ax = plt.subplots()
gr = (student_info.groupby(['final_result', 'gender'])
                  .size()
                  .unstack(fill_value=0)
                  .reindex(['Withdrawn', 'Fail', 'Pass', 'Distinction']))
x, w = np.arange(len(gr)), 0.35
ax.bar(x - w/2, gr['F'], w, label='Female', color=COLORS[0], edgecolor='black', linewidth=1.2)
ax.bar(x + w/2, gr['M'], w, label='Male',   color=COLORS[1], edgecolor='black', linewidth=1.2)
ax.set_title('Gender Distribution by Final Result', fontweight='bold')
ax.set_xlabel('Final Result', fontweight='bold')
ax.set_ylabel('Number of Students', fontweight='bold')
ax.set_xticks(x);  ax.set_xticklabels(gr.index)
ax.legend(fontsize=15, frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
plt.tight_layout()
plt.savefig(f'{OUT}plot3_gender_distribution.png', dpi=DPI, bbox_inches='tight')
plt.close();  print("  ✓ Plot 3  – Gender Distribution")

# ── Plot 4 : Score Distribution Box-plot ────────────────────
fig, ax = plt.subplots()
order = ['Withdrawn', 'Fail', 'Pass', 'Distinction']
data_bp = [df2[df2['final_result'] == r]['avg_score'].dropna().values for r in order]
bp = ax.boxplot(data_bp, labels=order, patch_artist=True,
                medianprops=dict(color='black', linewidth=2.5),
                whiskerprops=dict(linewidth=1.5),
                capprops=dict(linewidth=1.5),
                flierprops=dict(marker='o', alpha=0.3, markersize=4))
for patch, color in zip(bp['boxes'], COLORS[:4]):
    patch.set_facecolor(color);  patch.set_alpha(0.85)
ax.set_title('Score Distribution by Final Result', fontweight='bold')
ax.set_xlabel('Final Result', fontweight='bold')
ax.set_ylabel('Average Score (%)', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
plt.tight_layout()
plt.savefig(f'{OUT}plot4_score_boxplot.png', dpi=DPI, bbox_inches='tight')
plt.close();  print("  ✓ Plot 4  – Score Boxplot")

# ── Plot 5 : Top-15 Feature Importances ─────────────────────
fig, ax = plt.subplots()
imp = (pd.Series(rf2.feature_importances_, index=feat_cols)
         .sort_values(ascending=False)[:15])
bar_colors = [COLORS[i % len(COLORS)] for i in range(15)]
ax.barh(imp.index[::-1], imp.values[::-1],
        color=bar_colors[::-1], edgecolor='black', linewidth=0.8)
ax.set_title('Top 15 Feature Importances (Random Forest)', fontweight='bold')
ax.set_xlabel('Importance Score', fontweight='bold')
ax.set_ylabel('Feature', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
plt.tight_layout()
plt.savefig(f'{OUT}plot5_feature_importance.png', dpi=DPI, bbox_inches='tight')
plt.close();  print("  ✓ Plot 5  – Feature Importance")

# ── Plot 6 : Confusion Matrix ────────────────────────────────
fig, ax = plt.subplots()
cm = confusion_matrix(y_te, y_pred)
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(f'Confusion Matrix  (Acc: {acc*100:.2f}%)', fontweight='bold')
ax.set_xlabel('Predicted Label', fontweight='bold')
ax.set_ylabel('True Label', fontweight='bold')
ax.set_xticks(range(4));  ax.set_yticks(range(4))
ax.set_xticklabels(CLASS_NAMES, rotation=20, ha='right', fontsize=14)
ax.set_yticklabels(CLASS_NAMES, fontsize=14)
thresh = cm.max() / 2
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                fontweight='bold', fontsize=14,
                color='white' if cm[i, j] > thresh else 'black')
ax.grid(False)
plt.tight_layout()
plt.savefig(f'{OUT}plot6_confusion_matrix.png', dpi=DPI, bbox_inches='tight')
plt.close();  print("  ✓ Plot 6  – Confusion Matrix")

# ── Plot 7 : Multi-class ROC Curves ─────────────────────────
fig, ax = plt.subplots()
y_bin = label_binarize(y_te, classes=[0, 1, 2, 3])
for i, cname in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    ra = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=COLORS[i], lw=2.5, label=f'{cname}  (AUC={ra:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5)
ax.set_xlim([0, 1]);  ax.set_ylim([0, 1.02])
ax.set_title('Multi-Class ROC Curves (Voting Ensemble)', fontweight='bold')
ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate', fontweight='bold')
ax.legend(fontsize=13, frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
plt.tight_layout()
plt.savefig(f'{OUT}plot7_roc_curves.png', dpi=DPI, bbox_inches='tight')
plt.close();  print("  ✓ Plot 7  – ROC Curves")

# ── Plot 8 : 5-Fold Cross-Validation ────────────────────────
fig, ax = plt.subplots()
fold_labels = [f'Fold {i+1}' for i in range(5)]
bars = ax.bar(fold_labels, cv_scores * 100,
              color=COLORS[:5], edgecolor='black', linewidth=1.2, width=0.5)
ax.axhline(cv_scores.mean() * 100, color='crimson', linestyle='--', linewidth=2,
           label=f'Mean = {cv_scores.mean()*100:.2f}%')
for b, v in zip(bars, cv_scores):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.05,
            f'{v*100:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=15)
ax.set_title('5-Fold Cross-Validation Accuracy', fontweight='bold')
ax.set_xlabel('Fold', fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_ylim([cv_scores.min() * 100 - 2, 101])
ax.legend(fontsize=15, frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
plt.tight_layout()
plt.savefig(f'{OUT}plot8_cross_validation.png', dpi=DPI, bbox_inches='tight')
plt.close();  print("  ✓ Plot 8  – Cross-Validation")

# ── Plot 9 : Age Band vs Result (Stacked %) ─────────────────
fig, ax = plt.subplots()
ar = (student_info.groupby(['age_band', 'final_result'])
                  .size()
                  .unstack(fill_value=0))
order_cols = [c for c in ['Withdrawn', 'Fail', 'Pass', 'Distinction'] if c in ar.columns]
ar = ar[order_cols]
ar_pct = ar.div(ar.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(ar_pct))
for i, col in enumerate(ar_pct.columns):
    ax.bar(ar_pct.index, ar_pct[col], bottom=bottom,
           label=col, color=COLORS[i], edgecolor='black', linewidth=0.8)
    bottom += ar_pct[col].values
ax.set_title('Result Distribution by Age Band', fontweight='bold')
ax.set_xlabel('Age Band', fontweight='bold')
ax.set_ylabel('Percentage (%)', fontweight='bold')
ax.legend(fontsize=13, frameon=False, loc='upper right')
ax.set_ylim([0, 115])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
plt.tight_layout()
plt.savefig(f'{OUT}plot9_age_result_stacked.png', dpi=DPI, bbox_inches='tight')
plt.close();  print("  ✓ Plot 9  – Age Band Stacked Bar")

# ── Plot 10 : Model Accuracy Comparison ─────────────────────
fig, ax = plt.subplots()
model_names = ['Decision\nTree', 'Random\nForest', 'Extra\nTrees', 'Voting\nEnsemble']
model_accs  = [acc_dt * 100, acc_rf * 100, acc_et * 100, acc * 100]
bars = ax.bar(model_names, model_accs,
              color=COLORS[:4], edgecolor='black', linewidth=1.2, width=0.5)
for b, v in zip(bars, model_accs):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.1,
            f'{v:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=16)
ax.axhline(98, color='crimson', linestyle='--', linewidth=2, label='98% Threshold')
ax.set_title('Model Accuracy Comparison', fontweight='bold')
ax.set_xlabel('Model', fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_ylim([min(model_accs) - 3, 101])
ax.legend(fontsize=14, frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(False)
plt.tight_layout()
plt.savefig(f'{OUT}plot10_model_comparison.png', dpi=DPI, bbox_inches='tight')
plt.close();  print("  ✓ Plot 10 – Model Comparison")

# ─── SUMMARY ────────────────────────────────────────────────
print(f"""
╔══════════════════════════════════════════╗
║         FINAL RESULTS SUMMARY           ║
╠══════════════════════════════════════════╣
║  Voting Ensemble  : {acc*100:.2f}%             ║
║  CV Mean Accuracy : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%   ║
║  Random Forest    : {acc_rf*100:.2f}%             ║
║  Extra Trees      : {acc_et*100:.2f}%             ║
║  Decision Tree    : {acc_dt*100:.2f}%             ║
║  All 10 Plots ✓                         ║
╚══════════════════════════════════════════╝
""")